In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
%matplotlib inline
%matplotlib qt
import matplotlib.pyplot as plt
import torch.optim as optim


In [2]:
import torchvision
import torchvision.transforms as transforms
import torch

train_set = torchvision.datasets.FashionMNIST(
    root="./data/FashionMNIST",
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

test_set = torchvision.datasets.FashionMNIST(
    root="./data/FashionMNIST",
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

index = 0

image, label = train_set[index]

# -----------------------------------
# Print info
# -----------------------------------

print("Training Samples:", len(train_set))
print("Test Samples:", len(test_set))
print("Index:", index)
print("Label:", label)
print("Tensor shape:", image.shape)

Training Samples: 60000
Test Samples: 10000
Index: 0
Label: 9
Tensor shape: torch.Size([1, 28, 28])


In [5]:
batch_size = 100

train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_set,
    batch_size=batch_size,
    shuffle=False
)

In [6]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np

class Network(nn.Module):
    def __init__(self):
        super(Network, self).__init__()
        
        #Layer takes one input, has a kernel/filter of size 5 and outputs 5 featuremaps
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5)
        #Next layer taking in input of val 1
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=12, kernel_size=5)
        
        self.fc1 = nn.Linear(in_features=12*4*4, out_features=120)
        self.fc2 = nn.Linear(in_features=120, out_features=60)
        self.out = nn.Linear(in_features=60, out_features=10)

    def forward(self, t):
        #Implement forward pass
        t=t
        
        t=self.conv1(t)
        t=F.relu(t)
        t=F.max_pool2d(t, kernel_size=2, stride=2)
        
        t=self.conv2(t)
        t=F.relu(t)
        t=F.max_pool2d(t, kernel_size=2, stride=2)
        
        
        t=t.reshape(-1,12 * 4 * 4)
        t=self.fc1(t)
        t=F.relu(t)
        
        t=self.fc2(t)
        t=F.relu(t)
        
        t=self.out(t)
       # t=F.softmax(t, dim=1)
        
        return t

In [11]:
from itertools import product

parameters = dict(
    lr=[.01,.001],
    batch_size=[10,100,1000],
    shuffle=[True,False]
)

paramValues = [
    v for v in parameters.values()
]

for lr, batch_size, shuffle in product(*paramValues):
    print(
        lr,
        batch_size,
        shuffle
    )

0.01 10 True
0.01 10 False
0.01 100 True
0.01 100 False
0.01 1000 True
0.01 1000 False
0.001 10 True
0.001 10 False
0.001 100 True
0.001 100 False
0.001 1000 True
0.001 1000 False


In [12]:
network = Network()
optimizer = optim.Adam(network.parameters(), lr=0.01)
#Pred Shape: torch.Size([10, 10]) --> Two axis each length 10, ten images with 10 prediction classes

#Argmax to check which index has highest prediction value == compare with label after
#pred.argmax(dim=1)

#Compares with the label and gives 1 or 0 for match & calling sum reduces the output into a single number of correct predictions
#pred.argmax(dim=1).eq(labels).sum()

#Same thing but uses item to get the number of correct predictions
def get_num_correct(prediction, label):
    return prediction.argmax(dim=1).eq(label).sum().item()

In [14]:
from torch.utils.tensorboard import SummaryWriter
images, labels = next(iter(train_loader))

grid = torchvision.utils.make_grid(images)

comment = f"batch_size={batch_size}_lr={lr}"

tb = SummaryWriter(comment=comment)

tb.add_image(
    "FashionMNIST Images",
    grid
)

tb.add_graph(
    network,
    images
)

In [16]:
#Training loop
batch_size = 100
lr = 0.01
network = Network()

training_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_set,
    batch_size=batch_size,
    shuffle=False
)

optimizer = optim.Adam(network.parameters(), lr=lr)

images, labels = next(iter(training_loader))
grid = torchvision.utils.make_grid(images)

comment = f'batch_size={batch_size}, lr = {lr}'
tb= SummaryWriter(comment=comment)

tb.add_image('images', grid)
tb.add_graph(network, images)

for epoch in range(5):

    network.train()

    totalLoss = 0
    totalCorrect = 0
    
    for batch in training_loader:
        images, labels = batch
        
        pred = network(images)
        loss = F.cross_entropy(pred, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        totalLoss += loss.item() * batch_size
        totalCorrect += get_num_correct(pred, labels)

    network.eval()

    testCorrect = 0

    with torch.no_grad():

        for batch in test_loader:
            images, labels = batch

            pred = network(images)

            testCorrect += get_num_correct(
                pred,
                labels
            )

    testAccuracy = testCorrect / len(test_set)
    
    tb.add_scalar('Loss:', totalLoss, epoch)
    tb.add_scalar('Correct:', totalCorrect, epoch)
    tb.add_scalar('Accuracy:', totalCorrect/len(train_set), epoch)
    tb.add_scalar('Accuracy/Test:', testAccuracy, epoch)
    
    tb.add_histogram('conv1.bias', network.conv1.bias, epoch)
    tb.add_histogram('conv1.weight',network.conv1.weight, epoch)
    tb.add_histogram('conv1.weight.grad', network.conv1.weight.grad, epoch)

    print(
        "epoch:",
        epoch,
        "Total Correct:",
        totalCorrect,
        "Train Accuracy:",
        totalCorrect/len(train_set),
        "Test Accuracy:",
        testAccuracy,
        "Loss:",
        totalLoss
    )

tb.close()

epoch: 0 Total Correct: 46684 Train Accuracy: 0.7780666666666667 Test Accuracy: 0.8442 Loss: 35672.020599246025
epoch: 1 Total Correct: 51320 Train Accuracy: 0.8553333333333333 Test Accuracy: 0.8586 Loss: 23144.75508481264
epoch: 2 Total Correct: 52122 Train Accuracy: 0.8687 Test Accuracy: 0.8564 Loss: 21048.81298393011
epoch: 3 Total Correct: 52454 Train Accuracy: 0.8742333333333333 Test Accuracy: 0.8683 Loss: 20265.53690880537
epoch: 4 Total Correct: 52764 Train Accuracy: 0.8794 Test Accuracy: 0.8802 Loss: 19403.145569562912


In [17]:
torch.save(
    network.state_dict(),
    "fashion_cnn.pth"
)

print("Model Saved")

Model Saved


In [26]:
all_preds = torch.tensor([])

network.eval()

with torch.no_grad():

    for batch in test_loader:

        images, labels = batch

        preds = network(images)

        all_preds = torch.cat(
            (all_preds, preds),
            dim=0
        )

In [27]:
preds_correct = get_num_correct(
    all_preds,
    test_set.targets
)

print(
    'Correct:',
    preds_correct
)

print(
    'Accuracy:',
    preds_correct / len(test_set)
)

Correct: 8802
Accuracy: 0.8802


In [28]:
stacked = torch.stack(
    (
        test_set.targets,
        all_preds.argmax(dim=1)
    ),
    dim=1
)

cmt = torch.zeros(
    10,
    10,
    dtype=torch.int32
)

for p in stacked:
    j, k = p.tolist()
    cmt[j, k] = cmt[j, k] + 1

cmt

tensor([[886,   0,  17,  23,   3,   2,  63,   0,   6,   0],
        [  4, 967,   0,  18,   2,   0,   6,   0,   3,   0],
        [ 21,   2, 816,   8, 104,   0,  48,   0,   1,   0],
        [ 35,   6,  12, 902,  26,   0,  18,   0,   1,   0],
        [  1,   0,  81,  41, 827,   0,  49,   0,   1,   0],
        [  0,   0,   0,   0,   0, 980,   0,  16,   0,   4],
        [228,   2,  98,  23,  89,   0, 547,   1,  12,   0],
        [  0,   0,   0,   0,   0,  21,   0, 967,   0,  12],
        [  1,   0,   4,   2,   6,   5,   4,   7, 971,   0],
        [  0,   0,   0,   0,   0,  12,   0,  49,   0, 939]], dtype=torch.int32)

In [29]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))

sns.heatmap(
    cmt,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('FashionMNIST Confusion Matrix')

plt.show()

In [30]:
network.eval()

with torch.no_grad():

    images, labels = next(iter(test_loader))

    preds = network(images)

    preds = preds.argmax(dim=1)

incorrect = preds.ne(labels)

print(
    "Incorrect Predictions:",
    incorrect.sum().item()
)

Incorrect Predictions: 18
